## Advanced_CI_w5
### CH9

### 9.1 온라인 마케팅 데이터셋

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("online_mkt.csv").astype({"date":"datetime64[ns]"})
df.head()

,app_download,population,city,state,date,post,treated
0,3066.0,12396372,sao_paulo,sao_paulo,2022-03-01,0,1
1,2701.0,12396372,sao_paulo,sao_paulo,2022-03-02,0,1
2,1927.0,12396372,sao_paulo,sao_paulo,2022-03-03,0,1
3,1451.0,12396372,sao_paulo,sao_paulo,2022-03-04,0,1
4,1248.0,12396372,sao_paulo,sao_paulo,2022-03-05,0,1


In [8]:
treated = list(df.query("treated==1")["city"].unique())
treated

['sao_paulo', 'porto_alegre', 'joao_pessoa']

In [9]:
df_norm = df.assign(app_download_pct = 100*df["app_download"]/df["population"])
df_norm.head()

,app_download,population,city,state,date,post,treated,app_download_pct
0,3066.0,12396372,sao_paulo,sao_paulo,2022-03-01,0,1,0.024733
1,2701.0,12396372,sao_paulo,sao_paulo,2022-03-02,0,1,0.021789
2,1927.0,12396372,sao_paulo,sao_paulo,2022-03-03,0,1,0.015545
3,1451.0,12396372,sao_paulo,sao_paulo,2022-03-04,0,1,0.011705
4,1248.0,12396372,sao_paulo,sao_paulo,2022-03-05,0,1,0.010067


In [10]:
tr_period = df_norm.query("post==1")["date"].min() # post==1인 가장 빠른 날짜
tr_period

Timestamp('2022-05-01 00:00:00')

### 9.2 행렬 표현

In [11]:
def reshape_sc_data(df: pd.DataFrame,
                    geo_col: str,
                    time_col: str,
                    y_col: str,
                    tr_geos: str,
                    tr_start: str):
    df_pivot = df.pivot(index=time_col, columns=geo_col, values=y_col)

    y_co = df_pivot.drop(columns=tr_geos)
    y_tr = df_pivot[tr_geos]

    y_pre_co = y_co[df_pivot.index < tr_start]
    y_pre_tr = y_tr[df_pivot.index < tr_start]

    y_post_co = y_co[df_pivot.index >= tr_start]
    y_post_tr = y_tr[df_pivot.index >= tr_start]

    return y_pre_co, y_pre_tr, y_post_co, y_post_tr

In [12]:
y_pre_co, y_pre_tr, y_post_co, y_post_tr = reshape_sc_data(
    df_norm,
    geo_col="city",
    time_col="date",
    y_col="app_download_pct",
    tr_geos=treated,
    tr_start=str(tr_period)
)

y_pre_tr.head()

city,sao_paulo,porto_alegre,joao_pessoa
date,,,
2022-03-01,0.024733,0.004288,0.022039
2022-03-02,0.021789,0.008107,0.020344
2022-03-03,0.015545,0.004891,0.012352
2022-03-04,0.011705,0.002948,0.018285
2022-03-05,0.010067,0.006767,0.000000


### 9.3 통제집단합성법과 수평 회귀분석

In [13]:
from sklearn.linear_model import LinearRegression # 통제집단의 가중치를 추정하는 것이 목표, smf.ols보다 적절함

model = LinearRegression(fit_intercept=False)
model.fit(y_pre_co, y_pre_tr.mean(axis=1))

# 가중치 추출
weight_lr = model.coef_
weight_lr.round(3)

array([-0.65 , -0.058, -0.239,  0.971,  0.03 , -0.204,  0.007,  0.095,
        0.102,  0.106,  0.074,  0.079,  0.032, -0.5  , -0.041, -0.154,
       -0.014,  0.132,  0.115,  0.094,  0.151, -0.058, -0.353,  0.049,
       -0.476, -0.11 ,  0.158, -0.002,  0.036, -0.129, -0.066,  0.024,
       -0.047,  0.089, -0.057,  0.429,  0.23 , -0.086,  0.098,  0.351,
       -0.128,  0.128, -0.205,  0.088,  0.147,  0.555,  0.229])

In [14]:
y0_tr_hat = y_post_co.dot(weight_lr) # model.predict(y_post_co)와 동일

In [15]:
att = y_post_tr.mean(axis=1) - y0_tr_hat
att

date
2022-05-01    0.001646
2022-05-02    0.007035
2022-05-03    0.006899
2022-05-04    0.004678
2022-05-05    0.000941
                ...   
2022-06-26   -0.000733
2022-06-27    0.001350
2022-06-28    0.000049
2022-06-29   -0.005465
2022-06-30   -0.004994
Length: 61, dtype: float64

### 9.4 표준 통제집단합성법

In [16]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import (check_X_y, check_array, check_is_fitted)
import cvxpy as cp

class SyntheticControl(BaseEstimator, RegressorMixin):
    def __init__(self,):
        pass

    def fit(self, y_pre_co, y_pre_tr):
        y_pre_co, y_pre_tr = check_X_y(y_pre_co, y_pre_tr)
        w = cp.Variable(y_pre_co.shape[1])

        objective = cp.Minimize(cp.sum_squares(y_pre_co@w - y_pre_tr))
        constraints = [cp.sum(w) == 1, w >=0]

        problem = cp.Problem(objective, constraints)

        self.loss_ = problem.solve(verbose=False)
        self.w_ = w.value

        self.is_fitted_ = True
        return self

    def predict(self, y_co):

        check_is_fitted(self)
        y_co = check_array(y_co)

        return y_co @ self.w_
    
        

In [17]:
model = SyntheticControl()
model.fit(y_pre_co, y_pre_tr.mean(axis=1))

# 가중치 구하기
model.w_.round(3)

array([-0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,  0.076,  0.037,
        0.083,  0.01 , -0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,
        0.061,  0.123,  0.008,  0.074, -0.   ,  0.   , -0.   , -0.   ,
       -0.   , -0.   , -0.   , -0.   , -0.   ,  0.   , -0.   ,  0.092,
       -0.   , -0.   ,  0.   ,  0.046,  0.089,  0.   ,  0.067,  0.061,
        0.   , -0.   , -0.   ,  0.088,  0.   ,  0.086, -0.   ])

### 9.5 통제집단합성법과 공변량

In [18]:
df_norm_cov = pd.read_csv("online_mkt_cov.csv").astype({"date":"datetime64[ns]"})
df_norm_cov.head()

,app_download,population,city,state,date,post,treated,app_download_pct,comp_download_pct
0,3066.0,12396372,sao_paulo,sao_paulo,2022-03-01,0,1,0.024733,0.026280
1,2701.0,12396372,sao_paulo,sao_paulo,2022-03-02,0,1,0.021789,0.023925
2,1927.0,12396372,sao_paulo,sao_paulo,2022-03-03,0,1,0.015545,0.018930
3,1451.0,12396372,sao_paulo,sao_paulo,2022-03-04,0,1,0.011705,0.015858
4,1248.0,12396372,sao_paulo,sao_paulo,2022-03-05,0,1,0.010067,0.014548


In [19]:
from toolz import partial

reshaper = partial(reshape_sc_data,
                df=df_norm_cov,
                geo_col = "city",
                time_col = "date",
                tr_geos = treated,
                tr_start = str(tr_period))

y_pre_co, y_pre_tr, y_post_co, y_post_tr = reshaper(y_col="app_download_pct")

x_pre_co, _, x_post_co, _ = reshaper(y_col = "comp_download_pct")

In [20]:
def find_w_given_vs(vs, x_co_list, y_tr_pre):
    X_times_v = sum([x*v for x, v in zip(x_co_list, vs)])

    model = SyntheticControl()
    model.fit(X_times_v, y_tr_pre)

    return {"loss": model.loss_, "w": model.w_}

find_w_given_vs([1,0],
                [y_pre_co, x_pre_co],
                y_pre_tr.mean(axis=1)).get("w").round(3)

array([-0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,  0.076,  0.037,
        0.083,  0.01 , -0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,
        0.061,  0.123,  0.008,  0.074, -0.   ,  0.   , -0.   , -0.   ,
       -0.   , -0.   , -0.   , -0.   , -0.   ,  0.   , -0.   ,  0.092,
       -0.   , -0.   ,  0.   ,  0.046,  0.089,  0.   ,  0.067,  0.061,
        0.   , -0.   , -0.   ,  0.088,  0.   ,  0.086, -0.   ])

In [21]:
from scipy.optimize import minimize

def v_loss(vs):
    return find_w_given_vs(vs,
                            [y_pre_co, x_pre_co],
                            y_pre_tr.mean(axis=1)).get("loss")
v_solution = minimize(v_loss, [0,0], method='L-BFGS-B')
v_solution.x

array([1.80162468, 0.00311645])

In [22]:
w_cov = find_w_given_vs(v_solution.x,
                        [y_pre_co, x_pre_co],
                        y_pre_tr.mean(axis=1)).get("w").round(3)
w_cov

array([-0.   , -0.   ,  0.   , -0.   , -0.   , -0.   ,  0.086,  0.   ,
        0.034, -0.   , -0.   ,  0.035, -0.   ,  0.   , -0.   ,  0.   ,
        0.016,  0.051,  0.031,  0.01 , -0.   ,  0.   ,  0.   ,  0.053,
       -0.   ,  0.   , -0.   , -0.   ,  0.   ,  0.   , -0.   ,  0.048,
        0.084,  0.007,  0.   ,  0.   ,  0.147,  0.   ,  0.023,  0.009,
       -0.   ,  0.16 ,  0.   ,  0.036,  0.   ,  0.169, -0.   ])

In [23]:
y0_hat = sum([x*v for x, v in zip([y_post_co, x_post_co], v_solution.x)]).dot(w_cov)
att = y_post_tr.mean(axis=1) - y0_hat
att

date
2022-05-01    0.002829
2022-05-02    0.003949
2022-05-03    0.004959
2022-05-04    0.004268
2022-05-05    0.000794
                ...   
2022-06-26    0.002241
2022-06-27    0.001918
2022-06-28   -0.002604
2022-06-29   -0.002976
2022-06-30    0.000170
Length: 61, dtype: float64

### 9.6 통제집단합성법과 편향 제거

In [24]:
def debias_sc_atts(y_pre_co, y_pre_tr, y_post_co, y_post_tr, K=3):

    block_size = int(min(np.floor(len(y_pre_tr)/K), len(y_post_tr)))
    blocks = np.split(y_pre_tr.index[-K*block_size:],K)

    def fold_effect(hold_out):
        model = SyntheticControl()
        model.fit(
            y_pre_co.drop(hold_out),
            y_pre_tr.drop(hold_out)
        )

        bias_hat = np.mean(y_pre_tr.loc[hold_out] - model.predict(y_pre_co.loc[hold_out]))
        
        y0_hat = model.predict(y_post_co)
        return (y_post_tr - y0_hat) - bias_hat

    return pd.DataFrame([fold_effect(block) for block in blocks]).T

In [26]:
deb_atts = debias_sc_atts(y_pre_co, y_pre_tr.mean(axis=1), y_post_co, y_post_tr.mean(axis=1), K=3)
deb_atts.head()

,0,1,2
date,,,
2022-05-01,0.003314,0.002475,0.003162
2022-05-02,0.003544,0.002844,0.003368
2022-05-03,0.004644,0.003698,0.004759
2022-05-04,0.004706,0.002866,0.003603
2022-05-05,0.000134,-0.000541,0.000173


### 9.7 추론

In [27]:
atts_k = deb_atts.mean(axis=0).values
att = np.mean(atts_k)

print("atts_k", atts_k)
print("ATT", att)


atts_k [0.00414872 0.00260513 0.00317687]
ATT 0.003310235951728967


In [ ]:
K = len(atts_k)
T0 = len(y_pre_co)
T1 = len(y_post_co)
block_size = min(np.floor(T0/K), T1)

se_hat = np.sqrt(1+((K*block_size)/T1))*np.std(atts_k, ddof=1)/np.sqrt(K) # ddof=1 ; N-1로 나누는 sd
print("SE:", se_hat)

SE: 0.0006345693279981852


In [35]:
from scipy.stats import t
alpha = 0.1

[float(att - t.ppf(1-alpha/2, K-1)*se_hat),
float(att + t.ppf(1-alpha/2, K-1)*se_hat)]

[0.001457302664238376, 0.005163169239219558]

### 9.8 합성 이중차분법

In [37]:
sc_model = SyntheticControl()
sc_model.fit(y_pre_co, y_pre_tr.mean(axis=1))

float((y_post_tr.mean(axis=1) - sc_model.predict(y_post_co)).mean())

0.0033270409793961205

In [38]:
unit_w = pd.DataFrame(zip(y_pre_co.columns, sc_model.w_), columns=["city", "unit_weight"])
unit_w.head()

,city,unit_weight
0,ananindeua,-0.000008
1,aparecida_de_goiania,-0.000001
2,aracaju,-0.000008
3,belem,-0.000012
4,belford_roxo,-0.000006


In [39]:
df_with_w = (df_norm
            .assign(tr_post = lambda d: d["post"]*d["treated"])
            .merge(unit_w, on=["city"], how="left")
            .fillna({"unit_weight": df_norm["treated"].mean()}))
df_with_w.head()

,app_download,population,city,state,date,post,treated,app_download_pct,tr_post,unit_weight
0,3066.0,12396372,sao_paulo,sao_paulo,2022-03-01,0,1,0.024733,0,0.06
1,2701.0,12396372,sao_paulo,sao_paulo,2022-03-02,0,1,0.021789,0,0.06
2,1927.0,12396372,sao_paulo,sao_paulo,2022-03-03,0,1,0.015545,0,0.06
3,1451.0,12396372,sao_paulo,sao_paulo,2022-03-04,0,1,0.011705,0,0.06
4,1248.0,12396372,sao_paulo,sao_paulo,2022-03-05,0,1,0.010067,0,0.06


In [45]:
import statsmodels.formula.api as smf

mod = smf.wls( # weighted least squares
    "app_download_pct ~ tr_post + C(date)",
    data = df_with_w.query("unit_weight>=1e-10"),
    weights = df_with_w.query("unit_weight>=1e-10")["unit_weight"]
)
float(mod.fit().params["tr_post"])

0.0033293800074678313

In [54]:
class SyntheticControl(BaseEstimator, RegressorMixin):
    def __init__(self,fit_intercept=False):
        self.fit_intercept = fit_intercept

    def fit(self, y_pre_co, y_pre_tr):
        y_pre_co, y_pre_tr = check_X_y(y_pre_co, y_pre_tr)
        intercept = np.ones((y_pre_co.shape[0], 1))*self.fit_intercept
        X = np.concatenate([intercept, y_pre_co], axis = 1)
        w = cp.Variable(X.shape[1])
        
        objective = cp.Minimize(cp.sum_squares(X@w - y_pre_tr))
        constraints = [cp.sum(w[1:]) == 1, w[1:]>=0]
        
        problem = cp.Problem(objective, constraints)
        
        self.loss_ = problem.solve(verbose=False)
        self.intercept_ = float(w.value[0]) if self.fit_intercept else 0.0
        self.w_ = w.value[1:]

        self.is_fitted_ = True
        return self

    def predict(self, y_co):

        check_is_fitted(self)
        y_co = check_array(y_co)

        return y_co @ self.w_
    

In [56]:
time_sc = SyntheticControl(fit_intercept=True) # 절편을 포함한 모델 적합

time_sc.fit(y_pre_co.T, y_post_co.mean(axis=0))

time_w = pd.DataFrame(zip(y_pre_co.index, time_sc.w_), columns = ["date", "time_weight"])
time_w.tail()

,date,time_weight
56,2022-04-26,-0.000011
57,2022-04-27,0.071965
58,2022-04-28,-0.000002
59,2022-04-29,0.078350
60,2022-04-30,0.000002


In [57]:
scdid_df = (
    df_with_w
    .merge(time_w, on=["date"], how="left")
    .fillna({"time_weight": df_norm["post"].mean()})
    .assign(weight = lambda d: d["time_weight"]*d["unit_weight"])
)

In [60]:
did_model = smf.wls(
"app_download_pct ~ treated*post",
data = scdid_df.query("weight>1e-10"),
weights=scdid_df.query("weight>1e-10")["weight"]).fit()

float(did_model.params["treated:post"])

0.004086196404101924